In [4]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

True

## Basics

In [5]:
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(
    model=os.getenv("LLM_CHAT_MODEL"),
    base_url=os.getenv("LLM_HOST", "ollama"),
    temperature=0.7,
    max_tokens=1024,
)

llm.invoke("What is the capital of France?")

' The capital of France is Paris.'

In [6]:
from langchain_ollama import OllamaEmbeddings

embed = OllamaEmbeddings(
    model=os.getenv("LLM_EMBEDDING_MODEL", "mistral"),
    base_url=os.getenv("LLM_HOST", "ollama"),
)
input_text = "The meaning of life is 42"
vector = embed.embed_query(input_text)
print(vector)

[0.008596968, -0.00857477, -0.1530883, -0.04892, 0.000113009555, 0.048992712, -0.018965084, -0.024950992, 0.024965463, -0.003885075, -0.0700754, -0.008206605, 0.07211211, -0.013995139, 0.044751044, -0.046136335, -0.0113546355, -0.03657377, 0.011329823, 0.053426888, -0.015829986, -0.013355182, -0.049997594, 0.044475023, 0.07644843, 0.04438359, 0.008079022, 0.012103978, -0.009466673, 0.009221455, 0.024788419, 0.023667673, 0.01708908, 0.030556597, -0.046355586, -0.01807598, 0.012595285, 0.087363124, 0.09747744, -0.011615173, -0.013588504, 0.0053340327, 0.039753854, -0.009899324, 0.027058812, -0.056696653, -0.010603691, 0.018895783, 0.06462974, -0.0019988827, -0.056817178, -0.026589133, -0.00024798975, 0.050125774, 0.06993049, 0.073964745, -0.06227787, 0.014358483, -0.04450465, 0.023075076, 0.064871795, 0.08560714, -0.04568741, 0.074322335, 0.03598197, -0.044302784, -0.014307881, 0.0133107845, -0.017964918, -0.01292798, 0.044784166, -0.09262171, 0.0018493152, 0.020708079, 0.0022917467, 0.0

In [ ]:
from pydantic import BaseModel, Field
from enum import Enum

class DocumentSourceType(str, Enum):
    JIRA = "JIRA"
    CONFLUENCE = "CONFLUENCE"

class BaseMetadata(BaseModel):
    source: str = Field(..., description="Source of the document")
    type: DocumentSourceType = Field(..., description="Type of the document source")
    last_updated: str = Field(..., description="Last updated timestamp")

class JiraMetadata(BaseMetadata):
    issue_key: str = Field(..., description="JIRA issue key")
    project_key: str = Field(..., description="JIRA project key")

class ConfluenceMetadata(BaseMetadata):
    page_id: str = Field(..., description="Confluence page ID")
    space_key: str = Field(..., description="Confluence space key")

### Neo4J Integraytion

In [12]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URL"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
)

## Atlassian Integration

### Confluence

In [14]:
from langchain_core.documents import Document

In [28]:
from langchain_community.document_loaders import ConfluenceLoader

def map_to_confluence_doc(doc: Document) -> Document:
    meta: ConfluenceMetadata = ConfluenceMetadata(
            source=doc.metadata.get("source"),
            type=DocumentSourceType.CONFLUENCE,
            last_updated=doc.metadata.get("when"),
            page_id=doc.metadata.get("id"),
            space_key=doc.metadata.get("source").split("/")[5],
        )
    return Document(page_content=doc.page_content, metadata=meta.model_dump())


confluence_loader = ConfluenceLoader(
    url=os.getenv("CONFLUENCE_URL"),
    username=os.getenv("CONFLUENCE_USERNAME"),
    api_key=os.getenv("CONFLUENCE_API_KEY"),
    space_key=os.getenv("CONFLUENCE_SPACES"),
    include_attachments=False,
    limit=50,
    
)
confluence_documents = [ map_to_confluence_doc(doc) for doc in confluence_loader.load() ]
print(confluence_documents[0])

page_content='#E3FCEF Welcome to your software project space! We've added some suggestions and placeholders. Everything is customizable. Get started with templates: Template - Product requirements Template - Meeting notes Template - Decision documentation Status Software Engineering Best Practices Software engineering is a discipline that involves the application of engineering principles to software development in a methodical way. Adopting best practices in software engineering can lead to more efficient development processes, higher quality software, and improved team collaboration. Below are some key best practices that every software engineer should consider. 1. Version Control Using version control systems (VCS) like Git is essential for managing changes to code. It allows multiple developers to work on the same project simultaneously without conflicts. Version control provides a history of changes, making it easier to track bugs and revert to previous versions if necessary. 2. C

### Jira

In [ ]:
from typing import AsyncIterator, Iterator

from langchain_core.document_loaders import BaseLoader

from atlassian import Jira

class JiraLoader:
    def __init__(self, url, username, api_key, projects, limit=50):
        self.url = url
        self.username = username
        self.api_key = api_key
        self.projects = projects.split(",")
        self.limit = limit
        self.client = Jira(
            url=self.url, 
            username=self.username, 
            password=self.api_key, 
            cloud=True
        )

    def load(self):
        all_issues = []
        for project in self.projects:
            issues = self.client.jql(
                f"project = {project}",
            )
            all_issues.extend(issues['issues'])

        documents = []
        for issue in all_issues:
            content = issue['fields']['description'] or ""
            if hasattr(issue['fields'], "comment") and issue['fields']['comment']['comments']:
                comments = "\n".join(
                    [comment['body'] for comment in issue['fields']['comment']['comments']]
                )
                content += "\n\nComments:\n" + comments
            metadata = JiraMetadata(
                source=f"{self.url}/browse/{issue['key']}",
                type=DocumentSourceType.JIRA,
                last_updated=issue[''],
                issue_key=issue['key'],
                project_key=
            )
            metadata = {
                "issue_key": issue['key'],
                "summary": issue['fields']['summary'],
                "url": f"{self.url}/browse/{issue['key']}",
            }
            documents.append(Document(page_content=content, metadata=metadata))
        return documents

In [ ]:
jira_loader = JiraLoader(
    url=os.getenv("JIRA_URL"),
    username=os.getenv("JIRA_USERNAME"),
    api_key=os.getenv("JIRA_API_KEY"),
    projects=os.getenv("JIRA_PROJECTS"),
    limit=50,
)
jira_documents = jira_loader.load()
print(jira_documents[0])

page_content='Defeating a Balrog, a creature from J.R.R. Tolkien's Middle-earth legendarium, is purely fictional and not applicable in real life. However, if we consider the Balrog in terms of its literary portrayal, here are some strategies that could potentially be used against it:

# Fire Resistance: The Balrogs were immune to fire, making conventional weapons like swords or bows ineffective. A weapon capable of causing damage without using fire would be necessary. In the book "The Silmarillion," Gandalf uses a phial filled with holy water (Morgul-knife) to weaken a Balrog, but it's not explicitly stated that this would defeat one on its own.
# Mental Strength: In the book "The Lord of the Rings," Gandalf displays immense mental strength during his battle against a Balrog in Moria. He successfully uses the Balrog's arrogance and overconfidence to outsmart it, ultimately leading to their mutual destruction in the abyss of Moria.
# Allies: In both "The Silmarillion" and "The Lord of t

## Minimal RAG example

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embed)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [ ]:
def index_documents(documents):
    # Add node to Neo4j if not yet exists


In [ ]:
all_splits = text_splitter.split_documents(confluence_documents + jira_documents)
_ = vector_store.add_documents(documents=all_splits)

In [ ]:
from langchain import hub
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

prompt = hub.pull("rlm/rag-prompt")

# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response}

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
response = graph.invoke({"question": "How can I write stable and good software?"})
print(response["answer"])
print(response["context"])

 To write stable and good software, consider the following practices:

1. Implement clear lane structures to differentiate between teams, products, or streams.
2. Use markers to highlight important dates and milestones on a timeline (months or weeks).
3. Provide additional information about items on your roadmap by linking bars to pages. You can use the Roadmap planner for this purpose.
[Document(id='a8bdced9-6888-4340-adcf-ba64f494e319', metadata={'title': 'Software Development', 'id': '98457', 'source': 'https://trisnol.atlassian.net/wiki/spaces/SD/overview', 'when': '2025-09-19T11:44:22.359Z'}, page_content='lanes to differentiate between teams, products or streams. markers to highlight important dates and milestones. a timeline showing months or weeks. You can provide more information about items on your roadmap by linking a bar to a page. To add the Roadmap planner: When editing type / Find Roadmap planner in the dropdown Select Insert To edit the Roadmap planner: Select the place